# mBERT fine-tuning for IMDb sentiment classification

Fine-tunes `bert-base-multilingual-cased` on 5,000 IMDb reviews and evaluates it on 1,000 held-out reviews (binary labels: 0 negative, 1 positive). See the README in this folder for the setup and the result.


In [ ]:
!pip install datasets transformers scikit-learn accelerate


## Data

The IMDb movie review dataset from Hugging Face. The notebook takes 1,000 reviews of the original train split as its test set and keeps the first 5,000 of the remaining reviews for training. The original test split is not used.


In [ ]:
from datasets import load_dataset

dataset = load_dataset('stanfordnlp/imdb')

# Hold out 1,000 reviews of the original train split as the test set,
# then keep the first 5,000 of the remaining reviews for training.
dataset = dataset['train'].train_test_split(test_size=1000, seed=42)
dataset['train'] = dataset['train'].select(range(5000))
dataset


DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 5000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1000
    })
})

A look at the data: the first review of each split, the first rows, and the label counts of the training set.


In [ ]:
import pandas as pd

train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

# First review of each split
print(dataset['train'][0])
print(dataset['test'][0])

print("Train Data Examples:")
print(train_df.head())
print("\nTest Data Examples:")
print(test_df.head())

print(f"\nNumber of positive labels in training set: {(train_df['label'] == 1).sum()}")
print(f"Number of negative labels in training set: {(train_df['label'] == 0).sum()}")


{'text': "Go see this movie for the gorgeous imagery of Andy Goldsworthy's sculptures, and treat yourself to a thoroughly eye-opening and relaxing experience. The music perfectly complements the footage, but never draws attention towards itself. Some commentators called the interview snippets with the artist a weak spot, but consider this: why would you expand on this in a movie, if you can read Andy's musings at length in his books, or attend one of his excellent lectures? This medium is much more suitable to show the ephemeral nature of the artist's works, and is used expertly in this respect.", 'label': 1}
{'text': 'There is no relation at all between Fortier and Profiler but the fact that both are police series about violent crimes. Profiler looks crispy, Fortier looks classic. Profiler plots are quite simple. Fortier\'s plot are far more complicated... Fortier looks more like Prime Suspect, if we have to spot similarities... The main character is weak and weirdo, but have "clairvo

## Tokenization

Reviews are tokenized with the mBERT tokenizer and padded or truncated to 128 tokens.


In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')


In [ ]:
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True, max_length=128)


tokenized_datasets = dataset.map(tokenize_function, batched=True)


## Training

The pretrained `bert-base-multilingual-cased` gets a new two-class classification head and is fine-tuned for 3 epochs with the Hugging Face `Trainer`. The warning about unused and newly initialised weights in the log is expected: the pretraining heads are dropped and the classifier head starts from random values.


In [ ]:
from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model = BertForSequenceClassification.from_pretrained('bert-base-multilingual-cased', num_labels=2)

training_args = TrainingArguments(
    output_dir='./results',
    eval_strategy='epoch',  # evaluates on the test set after every epoch; the final model is kept
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_steps=10,
    report_to='none',
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
)

trainer.train()


TrainOutput(global_step=939, training_loss=0.42103371820764673, metrics={'train_runtime': 517.5493, 'train_samples_per_second': 28.983, 'train_steps_per_second': 1.814, 'total_flos': 986666457600000.0, 'train_loss': 0.42103371820764673, 'epoch': 3.0})

## Evaluation

Precision, recall and F1 for the positive class on the 1,000 test reviews, then a CSV with the gold and predicted label of every test review.


In [ ]:
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score

predictions = trainer.predict(tokenized_datasets['test'])
predicted_labels = np.argmax(predictions.predictions, axis=1)
true_labels = tokenized_datasets['test']['label']

# Scores for the positive class (label 1)
print(f"Precision: {precision_score(true_labels, predicted_labels)}")
print(f"Recall: {recall_score(true_labels, predicted_labels)}")
print(f"F1-score: {f1_score(true_labels, predicted_labels)}")


Precision: 0.8264462809917356
Recall: 0.819672131147541
F1-score: 0.823045267489712


In [ ]:
import pandas as pd

# Test reviews with the gold label and the predicted label
results_df = pd.DataFrame({
    'text': tokenized_datasets['test']['text'],
    'gold_label': tokenized_datasets['test']['label'],
    'predicted_label': predicted_labels,
})
results_df.to_csv('prediction_results.csv', index=False)

results_df


                                                  test  gold_standard  \
0    There is no relation at all between Fortier an...              1   
1    This movie is a great. The plot is very true t...              1   
2    George P. Cosmatos' "Rambo: First Blood Part I...              0   
3    In the process of trying to establish the audi...              1   
4    Yeh, I know -- you're quivering with excitemen...              0   
..                                                 ...            ...   
995  I enjoyed this film very much.Tobey Maguire gi...              1   
996  It really impresses me that it got made. The d...              0   
997  Really an amazing pile of pap! <br /><br />A p...              0   
998  The US State Dept. would not like us to see th...              1   
999  I have to admit I have always found it difficu...              1   

     predicted_label  
0                  1  
1                  1  
2                  1  
3                  0  
4       